In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb

import sys
!{sys.executable} -m pip install xgboost

In [2]:
file_path = "/Users/danisanchez/Downloads/Waves_2023.txt" #buoy data 
df_buoy = pd.read_csv(file_path, sep=r'\s+')

fileName = "/Users/danisanchez/Downloads/LA_AQS_2023.csv" #epa aqs data
pd.read_csv(fileName)
df_epa = pd.read_csv(fileName)

In [3]:
print("EPA columns:", df_epa.columns.tolist())
print(df_epa.head(2))

print("\nBuoy columns:", df_buoy.columns.tolist())
print(df_buoy.head(2))

EPA columns: ['State Code', 'County Code', 'Site Number', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Duration Description', 'Pollutant Standard', 'Date (Local)', 'Year', 'Day In Year (Local)', 'Units of Measure', 'Exceptional Data Type', 'Nonreg Observation Count', 'Observation Count', 'Observation Percent', 'Nonreg Arithmetic Mean', 'Arithmetic Mean', 'Nonreg First Maximum Value', 'First Maximum Value', 'First Maximum Hour', 'AQI', 'Daily Criteria Indicator', 'Tribe Name', 'State Name', 'County Name', 'City Name', 'Local Site Name', 'Address', 'MSA or CBSA Name', 'Data Source']
   State Code  County Code  Site Number  Parameter Code  POC  Latitude  \
0           6           37         1103           68103    1  34.06659   
1           6           37         1103           42602    3  34.06659   

   Longitude  Datum           Parameter Name Duration Description  ... AQI  \
0 -118.22688  WGS84  Ambient Min Temperature              24 HOUR  ...   .   


In [4]:
#BUOY DATA

In [5]:
df_buoy = df_buoy.replace([99.0, 999.0, 9999.0], np.nan)
df_buoy = df_buoy.dropna(subset=["WVHT", "DPD", "APD", "MWD", "WTMP"])

buoy_features = ["DPD", "APD", "MWD", "WTMP"]
X_buoy = df_buoy[buoy_features]
y_buoy = df_buoy["WVHT"]

In [6]:
X_train_b, X_temp_b, y_train_b, y_temp_b = train_test_split(
    X_buoy, y_buoy, test_size=0.4, random_state=42
)
X_val_b, X_test_b, y_val_b, y_test_b = train_test_split(
    X_temp_b, y_temp_b, test_size=0.5, random_state=42
)

In [7]:
#EPA DATA

In [8]:
all_pollutants = df_epa["Parameter Name"].unique()
no2_name = [
    p for p in all_pollutants if "NO2" in p or "Nitrogen dioxide" in p
][0]
o3_name = [p for p in all_pollutants if "Ozone" in p or "O3" in p][0]
pm25_name = [p for p in all_pollutants if "PM2.5" in p or "PM25" in p][0]

In [9]:
df_filtered = df_epa[
    df_epa["Parameter Name"].isin([no2_name, o3_name, pm25_name])
]
df_pivoted = df_filtered.pivot_table(
    index=["Date (Local)", "Site Number"],
    columns="Parameter Name",
    values="Arithmetic Mean",
).dropna()

df_pivoted = df_pivoted.rename(
    columns={no2_name: "NO2", o3_name: "O3", pm25_name: "PM25"}
).reset_index()

In [10]:
df_pivoted["Date (Local)"] = pd.to_datetime(df_pivoted["Date (Local)"])
df_pivoted["Month"] = df_pivoted["Date (Local)"].dt.month


def get_season(month):
    if month in (6, 7, 8):
        return "summer"
    elif month in (12, 1, 2):
        return "winter"
    return None


df_pivoted["Season"] = df_pivoted["Month"].apply(get_season)
df_ml = df_pivoted.dropna(subset=["Season"]).copy()
df_ml["Target"] = df_ml["Season"].map({"summer": 1, "winter": 0})

In [11]:
epa_features = ["O3", "NO2", "PM25"]
X_epa = df_ml[epa_features]
y_epa = df_ml["Target"]

X_train_val, X_epa_test, y_train_val, y_epa_test = train_test_split(
    X_epa, y_epa, test_size=0.2, random_state=42
)
X_epa_train, X_epa_val, y_epa_train, y_epa_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

# 5. Scale features & fit your optimal RidgeClassifier(alpha=100.0)
scaler_epa = StandardScaler()
X_epa_train_scaled = scaler_epa.fit_transform(X_epa_train)
X_epa_test_scaled = scaler_epa.transform(X_epa_test)

final_ridge = RidgeClassifier(alpha=100.0)
final_ridge.fit(X_epa_train_scaled, y_epa_train)

print(
    "Verified Test Error:",
    round(1 - final_ridge.score(X_epa_test_scaled, y_epa_test), 4),
)

# 6. Compute Permutation Feature Importance & Plot Barchart
perm_epa = permutation_importance(
    final_ridge,
    X_epa_test_scaled,
    y_epa_test,
    n_repeats=30,
    random_state=42,
    n_jobs=-1,
)
sorted_idx = perm_epa.importances_mean.argsort()

NameError: name 'RidgeClassifier' is not defined

In [ ]:
#models/plot

In [ ]:
epa_linear = RidgeClassifier(alpha=100.0, random_state=42)
epa_linear.fit(X_epa_train_scaled, y_epa_train)

xgb_optimal = xgb.XGBRegressor(
    n_estimators=40, objective="reg:squarederror", random_state=42
)
xgb_optimal.fit(X_train_b, y_train_b)

dt_optimal = DecisionTreeRegressor(max_depth=6, random_state=42)
dt_optimal.fit(X_train_b, y_train_b)

In [ ]:
lr_optimal = LinearRegression()
lr_optimal.fit(X_train_b, y_train_b)

In [ ]:
#epa data
plt.figure(figsize=(7, 4))
plt.barh(
    np.array(epa_features)[sorted_idx],
    perm_epa.importances_mean[sorted_idx],
    xerr=perm_epa.importances_std[sorted_idx],
    color="teal",
    edgecolor="black",
)
plt.xlabel("Mean Decrease in Accuracy (Permutation Importance)")
plt.title(
    "Permutation Importance: EPA AQS Summer vs. Winter (RidgeClassifier,"
    " alpha=100)"
)
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

#buoy

buoy_models = {
    "XGBoost (n_estimators=40)": xgb_optimal,
    "Decision Tree": dt_optimal,
    "Linear Regression": lr_optimal,
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (name, model) in zip(axes, buoy_models.items()):
    perm = permutation_importance(
        model, X_test_b, y_test_b, n_repeats=30, random_state=42, n_jobs=-1
    )
    sorted_idx = perm.importances_mean.argsort()

    ax.barh(
        np.array(buoy_features)[sorted_idx],
        perm.importances_mean[sorted_idx],
        xerr=perm.importances_std[sorted_idx],
        color="steelblue",
        edgecolor="black",
    )
    ax.set_title(f"{name}\n(Test $R^2$ = {model.score(X_test_b, y_test_b):.3f})")
    ax.set_xlabel("Permutation Importance (Drop in $R^2$)")
    ax.grid(axis="x", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
print("Test error:", 1 - epa_linear.score(X_epa_test_scaled, y_epa_test))